In [1]:
!pip install langchain langchain-openai langchain-community chromadb tiktoken
!pip install pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/23

In [2]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')


In [60]:
from langchain_core.documents import Document

docs = [
    # Properties
    Document(page_content="3 bedroom house in Colombo 05, rent 80000 LKR, near school and supermarket"),
    Document(page_content="2 bedroom apartment in Kandy city, rent 50000 LKR, mountain view"),
    Document(page_content="Luxury villa in Galle with sea view, rent 150000 LKR, private pool"),
    Document(page_content="Small house in Negombo, rent 40000 LKR, near beach and airport"),
    Document(page_content="Modern apartment in Colombo 03, rent 90000 LKR, city center location"),

    # Platform knowledge
    Document(page_content="""
HomeLanka.lk is a digital real estate marketplace platform in Sri Lanka.
It connects property buyers, sellers, and renters.
Users can browse listings, search properties, and find homes based on location, price, and preferences.
The platform provides verified property listings and an easy-to-use search system.
""")
]

In [61]:
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

llm = ChatOpenAI(
    model="openai/gpt-3.5-turbo",
    temperature=0,
    base_url="https://openrouter.ai/api/v1"
)

In [62]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url="https://openrouter.ai/api/v1",
)

In [63]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)

splits = text_splitter.split_documents(docs)

In [64]:
from langchain_community.vectorstores import Chroma

vector_store = Chroma.from_documents(
    documents=splits,
    embedding=embedding_model,
    # persist_directory="./home_lanka_db"
)

In [43]:
retriever = vector_store.as_retriever()

In [65]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
You are HomeLanka AI, an intelligent real estate assistant.

Your job is to help users find properties in Sri Lanka using ONLY the given context.

Rules:
- Use ONLY the provided context to answer
- If the answer is not in the context, say: "I don't have enough information"
- Do NOT guess or hallucinate properties
- Be clear, helpful, and concise

When answering:
- List best matching properties
- Include price, location, and key features
- Give a short recommendation

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "User question: {input}")
])

In [66]:
# Updated imports for LangChain v1.0+
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Create the question-answering chain
qa_chain = create_stuff_documents_chain(llm, prompt)

# Create the RAG chain
rag_chain = create_retrieval_chain(retriever, qa_chain)

In [67]:
query = "Find me a cheap house in Colombo under 80k"

response = rag_chain.invoke({"input": query})

print(response)

{'input': 'Find me a cheap house in Colombo under 80k', 'context': [Document(metadata={}, page_content='3 bedroom house in Colombo 05, rent 80000 LKR, near school and supermarket'), Document(metadata={}, page_content='3 bedroom house in Colombo 05, rent 80000 LKR, near school and supermarket'), Document(metadata={}, page_content='3 bedroom house in Colombo 05, rent 80000 LKR, near school and supermarket'), Document(metadata={}, page_content='Modern apartment in Colombo 03, rent 90000 LKR, city center location')], 'answer': "I don't have enough information"}


In [68]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Improved contextualization prompt for HomeLanka AI
contextualize_system_prompt = """
You are an AI assistant for HomeLanka, a real estate platform in Sri Lanka.

Your task is to reformulate the user's latest question into a standalone search query.

Rules:
- Use chat history to understand context
- Keep property intent (price, location, type of house/apartment/villa)
- Include important details like budget, city, property type
- Do NOT answer the question
- Only return the rewritten query
"""

# Contextualize prompt with memory
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# 🔍 History-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_prompt
)

In [69]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_prompt = """
You are HomeLanka AI, a smart real estate assistant for Sri Lanka.

Your job is to help users with:
1. Property search (houses, apartments, villas)
2. Information about HomeLanka platform

RULES:
- If context contains relevant property information → use it to answer
- If question is about HomeLanka platform → answer using available context or general platform knowledge
- Do NOT invent property listings or fake data
- Be clear, helpful, and concise

When answering property questions:
- Show best matching properties
- Include price, location, and key features
- Give short recommendation

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "User question: {input}")
])

In [70]:
prompt

ChatPromptTemplate(input_variables=['chat_history', 'context', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.

In [71]:
qa_chain = create_stuff_documents_chain(llm, prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

In [72]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Initialize the store for session histories
store = {}

# Function to get the session history for a given session ID
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Create the conversational RAG chain with session history
conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [73]:
response = conversational_rag_chain.invoke(
    {"input": "What is the rent for a 3 bedroom house in Colombo 05?"},
    config={"configurable": {"session_id": "101"}},
)

print(response["answer"])

The rent for a 3 bedroom house in Colombo 05 is 80000 LKR. It is located near a school and supermarket. If you are looking for a convenient location with nearby amenities, this property could be a good option for you.


In [74]:
response = conversational_rag_chain.invoke(
    {"input": "what is homelanka.lk"},
    config={"configurable": {"session_id": "101"}},
)

print(response["answer"])

HomeLanka.lk is a digital real estate marketplace platform in Sri Lanka. It connects property buyers, sellers, and renters, allowing users to browse listings, search properties, and find homes based on location, price, and preferences. The platform provides verified property listings and an easy-to-use search system for a seamless real estate experience.


In [75]:
response = conversational_rag_chain.invoke(
    {"input": "what is rent price of Colombo 05 house"},
    config={"configurable": {"session_id": "101"}},
)

print(response["answer"])

The rent price of a 3 bedroom house in Colombo 05 is 80000 LKR. This property is conveniently located near a school and supermarket, making it a desirable option for those looking for a comfortable home in a convenient location.
